# Lesson 07 - திட்டமிடல் வடிவமைப்பு மாதிரி

இந்த நோட்புக் Microsoft Agent Framework ஐ பயன்படுத்தி AI முகவர்களுக்கான **திட்டமிடல் வடிவமைப்பு மாதிரியை** விளக்குகிறது.
நீங்கள் சிக்கலான பயண கோரிக்கைகளை கட்டமைக்கப்பட்ட உபபணிகளாகப் பிரித்து, அவற்றை நிபுணர் முகவர்களுக்கு ஒதுக்கி,
மற்றும் தயாரிக்கப்பட்ட திட்டத்தை செயல்படுத்துவது எப்படி என்பதை கற்றுக்கொள்வீர்கள் — இதுவே Pydantic மாடல்களால் இயக்கப்படும் கட்டமைக்கப்பட்ட வெளியீட்டை பயன்படுத்துகிறது.


## அமைப்பு


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## பணியை பிரித்தல்

பணியை பிரித்தல் என்பது திட்டமிடல் வடிவமைப்பு பழக்கவழக்கத்தின் بنیادی அம்சமாகும். ஒரு agent ஒருவரிடம் கடுமையான கோரிக்கையை முழுமையாக கையாள விரும்புவதற்குப் பதிலாக, நாம் பிரச்சனையை சிறிய, நன்கு வரையறுக்கப்பட்ட **சிறுபணிகளாக** பிரிக்கின்றோம். ஒவ்வொரு சிறுபணியும் ஒரு நிபுணர் agent (உதாரணமாக, விமானங்கள், விடுதிகள், நடவடிக்கைகள்)க்கு தெளிவான முன்னுரிமைகள் மற்றும் சார்பு வரிசையை உடையவாறு ஒதுக்கப்படுகிறது.

இந்த அணுகுமுறை பல நன்மைகளை வழங்குகிறது:
- **தெளிவு**: ஒவ்வொரு சிறுபணிக்கும் ஒரு கண்ணோட்ட பொறுப்பு உள்ளது.
- **சமகால வினைமுறை**: சுயாதீன சிறுபணிகள் ஒரே நேரத்தில் செயல்படலாம்.
- **நம்பகத்தன்மை**: தோல்விகள் தனி சிறுபணிகளுக்கு மட்டும் வரம்புகளில் இருக்கும்.
- **பட்ஜெட் கண்காணிப்பு**: செலவுகள் ஒவ்வொரு சிறுபணிக்கும் கணக்கிடப்பட்டு ஒருங்கிணைக்கப்படுகின்றன.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## கட்டமைக்கப்பட்ட பயனர் வெளியீட்டுடன் திட்டமிடும் முகவரியை உருவாக்குதல்

திட்டமிடும் முகவர் ஒரு **முன்னணி மேசை ஒருங்கிணைப்பாளராக** செயல்படுகிறது. ஒரு உயர் நிலை பயண கோரிக்கையைப் பெற்றவுடன் அது ஒரு கட்டமைக்கப்பட்ட `TravelPlan` உற்பத்தி செய்கிறது — கோரிக்கையை துணைப் பணிகளாக பிரிப்பது, முன்னுரிமைகளை அமைத்தல், மற்றும் சார்புகளைக் கண்டறிந்து, ஒரு கான்சீர் அல்லது செயலாக்க স্তரங்கள் பணியை நிறைவேற்ற முடியும்.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## நிபுணர் கருவிகளுடன் ஒரு திட்டத்தை செயல்படுத்துதல்

முன் டெஸ்க் முகவர் ஒரு கட்டமைக்கப்பட்ட திட்டத்தை உருவாக்கியவுடன், **கான்சியெர்ஜ் முகவர்** அதை செயல்படுத்தும்.  
ஒவ்வொரு நிபுணர் கருவியும் துணைபணியின் ஒரு பிரிவை கையாள்கிறது (பயணப் பறவைகள், ஓட்டல்கள், செயல்பாடுகள்). கான்சியெர்ஜ் திட்டத்தின் துணைப் பணிகளை சார்பு வரிசையில் திரும்பிப் பார்த்து, ஒவ்வொன்றையும் தொடர்புடைய கருவிக்கு அனுப்பும்.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## சுருப்பு

இந்த பாடத்தில் நீங்கள் AI முகவர்கள் కోసం **திட்டமிடல் வடிவமைப்பு முறை** பற்றி கற்றுக்கொண்டீர்கள்:

1. **பணி பகுப்பாய்வு** — முன்னணி மேசை திட்டமிடல் முகவர் ஒரு சிக்கலான பயண கோரிக்கையை Pydantic மாதிரிகள் மூலம் கட்டமைக்கப்பட்ட துணைப் பணிகளாக உடைத்து, ஒவ்வொன்றையும் முன்னுரிமைகள் மற்றும் சார்புகளை கொண்டு ஒரு நிபுணர் முகவருக்கு ஒதுக்குகிறது.
2. **கட்டமைக்கப்பட்ட வெளியீடு** — `response_format` ஐ வழங்குவதன் மூலம் முகவர் ஓர் செல்லுபடியாகும் `TravelPlan` பொருளை சுதந்திர வடிவக் கட்டுரைக்கு பதிலாக திருப்பி அளிக்கிறது, இதனால் கீழ்கண்ட செயலாக்கம் நம்பகமாகிறது.
3. **திட்டநடைமுறை செயலாக்கம்** — ஒரு கான்சியெர்ஜ் முகவர் நிபுணர் கருவிகள் (`book_flight`, `reserve_hotel`, `book_activity`) பயன்படுத்தி துணைப் பணிகளை முறையாக செயல்படுத்தி முடிவுகளை அதிவிடுத்தான்.

இந்த முறை *என்ன செய்ய வேண்டும்* (திட்டமிடல்) என்பதையும் *எப்படிச் செய்ய வேண்டும்* (செயலாக்கம்) என்பதையும் பிரித்து முகவர்களை மேலும் தொகுத்தமைக்கக்கூடிய, சோதிக்கக்கூடிய மற்றும் விரிவாக்க வசதியுள்ளவைகளாக்குகிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**குறிப்பு**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டது. துல்லியம் காக்க நாம் முயலினாலும், தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்க வாய்ப்பு உள்ளது. அசல் ஆவணம் அதன் சொந்த மொழியில் அதிகாரப்பூர்வமான ஆதாரமாகக் கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பு பயன்படுத்துவதால் ஏற்படும் எந்த தவறுத்திருத்தங்கள் அல்லது தவறான புரிதல்களுக்கு நாங்கள் பொறுப்பேற்க மாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
